In [91]:
# import libs
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

In [92]:
def load_dataset(filepath):
    print("=" * 60)
    print("LOAD DATASET")
    print("=" * 60)

    df = pd.read_csv(filepath).set_index("Timestamp")
    
    print(df.head())
    print("Shape of the dataset")
    print(df.shape)
    print("\nCheck for missing values")
    print(df.isnull().sum())
    print("\nFirst five rows:")
    print(df.head())
    print("\nDescriptive stats")
    print(df.describe())
    print("\nDataset Info")
    print(df.info())
    print("\nState Distribution:")
    print(df["Condition Label"].value_counts(normalize=True) * 100)

    return df

In [93]:
def features(df):
    print("\n" + "=" * 60)
    print("FEATURES")
    print("=" * 60)
    

    target = "Condition Label"

    df = df.drop(columns=target)
    # y = df[target]
    features_columns = df.columns.tolist()

    
    # print("Features Shape:", .shape)
    # print("Target Shape:", y.shape)
    print("features_columns:", features_columns)
    print(df.head())

    return df, features_columns

In [94]:
def features_scale(df):
    print("\n" + "=" * 60)
    print("Features Scale")
    print("=" * 60)
    
    scaler =  MinMaxScaler()
    scaled = scaler.fit_transform(df)

    print("Features Scaled Successfuly")
    
    return scaled, scaler

In [95]:
def create_sequences(scaled, window_size=20):
    print("\n" + "=" * 60)
    print("Creating Window")
    print("=" * 60)
    
    X, y = [], []
    
    for i in range(len(scaled) - window_size):
        X.append(scaled[i:i+window_size])
        y.append(scaled[i+window_size])
    
    return np.array(X), np.array(y)
   

In [96]:
def split(X, y):
    print("\n" + "=" * 60)
    print("Split Data")
    print("=" * 60)
    
    cutoff = int(len(X) * 0.8)

    X_train, X_test = X[:cutoff], X[cutoff:]
    y_train, y_test = y[:cutoff], y[cutoff:]

    print("X_train shape:", X_train.shape)
    print("y_train shape:", y_train.shape)
    print("X_test shape:", X_test.shape)
    print("y_test shape:", y_test.shape)

    return X_train, X_test, y_train, y_test

In [97]:
def build_model(X, X_train, y_train, X_test, y_test):
    print("\n" + "=" * 60)
    print("Build Model")
    print("=" * 60)
    model = Sequential([
        LSTM(64, input_shape=(X.shape[1], X.shape[2])),
        Dense(32, activation='relu'),
        Dense(3) 
    ])
    
    model.compile(optimizer='adam', loss='mse')
    history = model.fit( 
        X_train, y_train,
        epochs=20,
        batch_size=32,
        validation_data=(X_test, y_test)
    )
    print("MODEL TRAINED SUCCESSFULLY")
    return model

In [100]:
def evaluate_model(model,scaled, scaler, X_test, y_test):
    predictions = model.predict(X_test)
    full_pred = np.zeros((predictions.shape[0], scaled.shape[1]))
    full_pred[:, :3] = predictions

    predictions = scaler.inverse_transform(full_pred)[:, :3]
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    print(rmse)

In [101]:
def main():
    filepath = "../data/building_health_monitoring_model_dataset.csv"

    df = load_dataset(filepath)

    df, features_columns = features(df)

    scaled, scaler = features_scale(df)

    X, y = create_sequences(scaled, 20)
    print("X shape:", X.shape)
    print("y shape:", y.shape)

    X_train, X_test, y_train, y_test = split(X, y)

    model = build_model(X, X_train, y_train, X_test, y_test)

    evaluate_model(model,scaled, scaler, X_test, y_test)

    # X_resam, X_test, y_resam, y_test = split_data(X, y)

    # # X_resam, y_resam = resample(X_train, y_train, replace=True, random_state=42)

    # model = build_and_train_model(X_resam, y_resam, numerical_features, categorical_features, feature_columns)

    # mertics = evaluate_model(model, X_resam, X_test, y_resam, y_test)

    # save_model_artifact(model)


if __name__ == "__main__":
    main()

LOAD DATASET
                     Accel_X (m/s^2)  Strain (με)  Temp (°C)  Condition Label
Timestamp                                                                    
2025-04-19 00:00:00         0.149014    61.843849  23.704760                0
2025-04-19 00:00:01        -0.041479    82.792300  24.953195                0
2025-04-19 00:00:02         0.194307    91.727889  25.027025                0
2025-04-19 00:00:03         0.456909   137.753753  25.708946                0
2025-04-19 00:00:04        -0.070246   111.131062  22.949712                0
Shape of the dataset
(1000, 4)

Check for missing values
Accel_X (m/s^2)    0
Strain (με)        0
Temp (°C)          0
Condition Label    0
dtype: int64

First five rows:
                     Accel_X (m/s^2)  Strain (με)  Temp (°C)  Condition Label
Timestamp                                                                    
2025-04-19 00:00:00         0.149014    61.843849  23.704760                0
2025-04-19 00:00:01        -0.04147

C:\Users\USER\OneDrive\Desktop\structural-sentinel\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - loss: 0.0726 - val_loss: 0.0333
Epoch 2/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0270 - val_loss: 0.0272
Epoch 3/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0262 - val_loss: 0.0271
Epoch 4/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0259 - val_loss: 0.0269
Epoch 5/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.0257 - val_loss: 0.0269
Epoch 6/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.0259 - val_loss: 0.0271
Epoch 7/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0256 - val_loss: 0.0278
Epoch 8/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0258 - val_loss: 0.0267
Epoch 9/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0256 - val_loss: 0.0268
Epoch 10/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0255 - val_loss: 0.0270
Epoch 11/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0254 - val_loss: 0.0271
Epoch 12/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0256 - val_l